# Output conversion (nb3): segmentations → DICOM-SEG + radiomics  —  shared, model-agnostic

Final step of the harmonized Segmentator workflow (CPU VM). Consumes the **Boundary-B**
archive `segmentations.tar.lz4` from any model's nb2 and produces the same artifact set
for **every** model, because everything here is computed from
`(NIfTI multilabel mask + reference DICOM + SNOMED mapping)` — independent of the
inference engine:
1. **DICOM-SEG** via dcmqi `itkimage2segimage` (`--segmentationType labelmap`).
2. **Radiomics** (pyradiomics or Radiomics.jl, `radiomicsMethod`) per label; feature classes from `radiomicsFeatureClasses` (default first-order + shape).
3. **DICOM SR (TID1500)** via dcmqi `tid1500writer` (`runStructuredReport`): one SR per SEG object encoding every feature with an IBSI quantity code + UCUM units (from `radiomicsFeatureCodesPath`, currently first-order + shape); uncoded features stay JSON-only.
4. Optional GCS upload and Healthcare API DICOM-store import.

Boundary-B layout consumed:
```
<SeriesInstanceUID>/<model>/segmentations/*.nii.gz   # multilabel volume(s)
<SeriesInstanceUID>/<model>/label_map.json           # {label_id: label_name}
```
The SEG core is ported from the MOOSE post-process notebook, generalized to key on the
per-run `label_map.json` sidecar so it is no longer MOOSE-specific; the SR writer is
ported from `TotalSegmentator/Notebooks/dicomsegAndRadiomicsSR_Notebook.ipynb`,
generalized to the canonical engine-neutral feature keys.

## Imports

In [ ]:
import csv
import json
import os
import re
import shutil
import subprocess
import time
import traceback
from pathlib import Path

import numpy as np
import nibabel as nib
from idc_index.index import IDCClient

NOTEBOOK_START = time.time()
def _elapsed(s=None):
    return f"{time.time() - (s if s is not None else NOTEBOOK_START):.1f}s"
print(f"[T+{_elapsed()}] Imports complete")

## Parameters

In [ ]:
segmentationArchivePath = "segmentations.tar.lz4"
snomedMappingPath = "snomed_mapping.csv"
modelName = ""
runRadiomics = True
runStructuredReport = True
radiomicsMethod = "pyradiomics"  # "pyradiomics" | "radiomicsjl" (one engine per run)
radiomicsFeatureClasses = "firstorder,shape"  # comma-separated; firstorder,shape,glcm,glrlm,glszm,ngtdm,gldm or "all"
# Feature -> DICOM quantity/units codes (IBSI + UCUM) for the TID1500 SR writer;
# WDL fetches workflows/common/resources/radiomicsFeaturesMaps.csv next to this
# notebook. Features without a coded row are JSON-only (no SR measurement).
radiomicsFeatureCodesPath = "radiomicsFeaturesMaps.csv"
radiomicsJlThreads = 1  # Julia threads; KEEP 1 -- Radiomics.jl multi-label extraction has a data race with >1 thread (0 = all vCPUs)
radiomicsMaxRoiMvox = 5.0  # skip radiomics for labels whose ROI exceeds this many Mvoxels (whole-body masks etc.); <= 0 disables the guard
dicomSegBucketUri = ""
dicomStoreImportUri = ""
inferenceUsageMetricsCsvPath = ""
convertUsageMetricsCsvPath = ""
input_uri = ""
secret_project = ""
runId = ""
# Optional GCS prefix for checkpoint/resume on preemption (same value as the inference
# task's checkpointGcsPath; namespaced by runId). Each finished series' DICOM-SEG +
# radiomics output is saved and a retried VM skips it. Empty = disabled.
checkpointGcs = ""

In [ ]:
# papermill's `-p` CLI flag only recognizes the Python-capitalized "True"/"False"
# literals (see papermill.cli._resolve_type); WDL's Boolean-to-String coercion
# always renders lowercase "true"/"false", which falls through unresolved as a
# (truthy) string. Normalize explicitly so a lowercase "false" from the WDL is
# never mistaken for a real True.
def _as_bool(v):
    return v if isinstance(v, bool) else str(v).strip().lower() in ('true', '1', 'yes')

runRadiomics = _as_bool(runRadiomics)
runStructuredReport = _as_bool(runStructuredReport)

# Radiomics engine selector. Normalize + validate; fall back to pyradiomics (with
# a warning) rather than hard-failing the whole output-conversion run on a typo.
RADIOMICS_METHODS = ('pyradiomics', 'radiomicsjl')
radiomicsMethod = str(radiomicsMethod).strip().lower()
if radiomicsMethod not in RADIOMICS_METHODS:
    print(f"WARNING: unknown radiomicsMethod {radiomicsMethod!r}; "
          f"falling back to 'pyradiomics' (valid: {RADIOMICS_METHODS})")
    radiomicsMethod = 'pyradiomics'

# Radiomics feature classes: engine-neutral (pyradiomics-style) names, applied
# to whichever engine is selected. Comma/whitespace separated, or "all".
# Canonical name -> (pyradiomics class name, Radiomics.jl feature symbol).
RADIOMICS_FEATURE_CLASS_MAP = {
    'firstorder': ('firstorder', 'first_order'),
    'shape':      ('shape',      'shape3d'),
    'glcm':       ('glcm',       'glcm'),
    'glrlm':      ('glrlm',      'glrlm'),
    'glszm':      ('glszm',      'glszm'),
    'ngtdm':      ('ngtdm',      'ngtdm'),
    'gldm':       ('gldm',       'gldm'),
}
DEFAULT_RADIOMICS_FEATURE_CLASSES = ('firstorder', 'shape')
_ALIASES = {'first_order': 'firstorder', 'shape3d': 'shape', 'shape_3d': 'shape'}

def _parse_feature_classes(raw):
    toks = [t for t in re.split(r'[,\s;]+', str(raw or '').strip().lower()) if t]
    if 'all' in toks:
        return tuple(RADIOMICS_FEATURE_CLASS_MAP)
    keep, unknown = [], []
    for t in toks:
        t = _ALIASES.get(t, t)
        if t in RADIOMICS_FEATURE_CLASS_MAP:
            if t not in keep:
                keep.append(t)
        else:
            unknown.append(t)
    if unknown:
        print(f"WARNING: unknown radiomics feature class(es) {unknown} ignored "
              f"(valid: {list(RADIOMICS_FEATURE_CLASS_MAP)} or 'all')")
    if not keep:
        print(f"WARNING: no valid radiomics feature classes in {raw!r}; "
              f"falling back to {DEFAULT_RADIOMICS_FEATURE_CLASSES}")
        keep = list(DEFAULT_RADIOMICS_FEATURE_CLASSES)
    return tuple(keep)

radiomicsFeatureClasses = _parse_feature_classes(radiomicsFeatureClasses)
print(f'Radiomics: method={radiomicsMethod} feature_classes={radiomicsFeatureClasses}')

## SNOMED mapping loader (ported from MOOSE post-process, keyed by label_name)

In [ ]:
def _code(designator, value, meaning):
    value = (value or '').strip()
    if not value:
        return None
    return {'CodingSchemeDesignator': (designator or '').strip(),
            'CodeValue': value, 'CodeMeaning': (meaning or '').strip()}

def _parse_rgb(raw):
    try:
        vals = [int(x) for x in str(raw).strip().strip('[]').split(',')]
        return vals if len(vals) == 3 else None
    except (ValueError, AttributeError):
        return None

def _load_snomed_csv(path):
    # NOTE: keyed on label_name alone (the CSV 'model' column is ignored). This
    # assumes a given label_name maps to the same SNOMED codes across every model
    # / sub-model — true for the current CSVs. If a future model introduces a
    # label_name with a conflicting code, key on (model, label_name) instead and
    # reconcile the engine model names emitted in label_map.json (e.g. moosez's
    # clin_ct_* vs the CSV's short names).
    _c, _t = 'SegmentedPropertyCategoryCodeSequence', 'SegmentedPropertyTypeCodeSequence'
    _tm = 'SegmentedPropertyTypeModifierCodeSequence'
    _ar, _arm = 'AnatomicRegionSequence', 'AnatomicRegionModifierSequence'
    mapping = {}
    with open(path, newline='') as f:
        for row in csv.DictReader(f):
            label = (row['label_name'] or '').strip()
            entry = {
                'category': _code(row[f'{_c}.CodingSchemeDesignator'], row[f'{_c}.CodeValue'], row[f'{_c}.CodeMeaning']),
                'type': _code(row[f'{_t}.CodingSchemeDesignator'], row[f'{_t}.CodeValue'], row[f'{_t}.CodeMeaning']),
                'type_modifier': _code(row[f'{_tm}.CodingSchemeDesignator'], row[f'{_tm}.CodeValue'], row[f'{_tm}.CodeMeaning']),
                'region': _code(row[f'{_ar}.CodingSchemeDesignator'], row[f'{_ar}.CodeValue'], row[f'{_ar}.CodeMeaning']),
                'region_modifier': _code(row[f'{_arm}.CodingSchemeDesignator'], row[f'{_arm}.CodeValue'], row[f'{_arm}.CodeMeaning']),
                'rgb': _parse_rgb(row['recommendedDisplayRGBValue']),
            }
            if label in mapping and mapping[label] != entry:
                raise ValueError(f"Conflicting SNOMED coding for label '{label}' in {path}")
            mapping[label] = entry
    return mapping

# label_to_snomed is loaded *after* the Boundary-B archive is extracted (see the
# "Extract Boundary-A archive" cell). This lets a model that bundles its own
# mapping into the archive (MOOSE ships moosez's authoritative
# moose_snomed_mapping.csv) take precedence over the WDL-fetched snomedMappingPath
# (used by TotalSegmentator, whose pinned engine version ships no SNOMED table).

## Helpers: dcmqi config, reference DICOM, labels

In [ ]:
idc_client = IDCClient()

# When input_uri is set (private GCS, non-IDC data) the reference DICOM is not in
# IDC, so it is staged from GCS once (see the "Stage reference DICOM" cell) into
# GCS_STAGED/<SeriesInstanceUID>/ and copied per-series by download_dicom below.
GCS_STAGED = Path('/tmp/gcs_ref_dicom')

def label_voxel_counts(nifti_path):
    """{label_id: voxel count} for every non-zero label in a NIfTI mask."""
    arr = np.asanyarray(nib.load(str(nifti_path)).dataobj)
    vals, counts = np.unique(arr, return_counts=True)
    return {int(v): int(c) for v, c in zip(vals, counts) if v != 0}

def unique_nonzero_labels(nifti_path):
    return sorted(label_voxel_counts(nifti_path))

def download_dicom(uid, dest):
    dest.mkdir(parents=True, exist_ok=True)
    if input_uri:
        # Reference DICOM was pre-staged from GCS (see staging cell).
        src = GCS_STAGED / uid
        if not src.is_dir():
            raise FileNotFoundError(f'No staged reference DICOM for {uid} under {input_uri}')
        for f in src.rglob('*.dcm'):
            shutil.copy(str(f), str(dest / f.name))
    else:
        idc_client.download_from_selection(downloadDir=str(dest), seriesInstanceUID=uid)

def find_series_number(dicom_dir):
    try:
        import pydicom
        for p in dicom_dir.rglob('*.dcm'):
            ds = pydicom.dcmread(str(p), stop_before_pixels=True)
            sn = getattr(ds, 'SeriesNumber', None)
            if sn is not None:
                return str(int(sn))
    except Exception:
        pass
    return '1'

def build_dcmqi_config(model, labels, label_names, series_number):
    """labels: list[int] present in the mask. label_names: {label_id: name} from
    the Boundary-B label_map.json sidecar. Every label must resolve to a complete
    SNOMED entry (category+type) or this raises — mapping gaps surface immediately
    rather than producing meaningless SEG metadata. recommendedDisplayRGBValue is
    optional (dcmqi does not require it), so a blank RGB does not disqualify a
    label."""
    segments, unmapped = [], []
    for label_id in labels:
        label_id = int(label_id)
        name = (label_names.get(str(label_id)) or label_names.get(label_id) or f'segment_{label_id}')
        rec = label_to_snomed.get(str(name).strip())
        if not (rec and rec.get('type') and rec.get('category')):
            unmapped.append((label_id, name))
            continue
        seg_type = rec['type']
        display_name = seg_type['CodeMeaning']
        if rec.get('type_modifier'):
            display_name = f"{display_name} ({rec['type_modifier']['CodeMeaning']})"
        segment = {
            'labelID': label_id,
            'SegmentDescription': display_name,
            'SegmentLabel': display_name,
            'SegmentAlgorithmType': 'AUTOMATIC',
            'SegmentAlgorithmName': model or (modelName or 'segmentator'),
            'SegmentedPropertyCategoryCodeSequence': rec['category'],
            'SegmentedPropertyTypeCodeSequence': seg_type,
        }
        if rec.get('rgb'):
            segment['recommendedDisplayRGBValue'] = rec['rgb']
        if rec.get('type_modifier'):
            segment['SegmentedPropertyTypeModifierCodeSequence'] = rec['type_modifier']
        if rec.get('region'):
            segment['AnatomicRegionSequence'] = rec['region']
            if rec.get('region_modifier'):
                segment['AnatomicRegionModifierSequence'] = rec['region_modifier']
        segments.append(segment)
    if unmapped:
        details = ', '.join(f"{lid} ('{n}')" for lid, n in unmapped)
        raise KeyError(f"No complete SNOMED mapping for model '{model}' label(s): {details}. "
                       f'Add them to {snomedMappingPath} before generating DICOM SEG.')
    return {
        'ContentCreatorName': 'CloudSegmentator',
        'ClinicalTrialSeriesID': 'Session1',
        'ClinicalTrialTimePointID': '1',
        'SeriesDescription': f'{model} Segmentation',
        'SeriesNumber': str(int(series_number) * 100 + 1) if series_number.isdigit() else '100',
        'InstanceNumber': '1',
        'BodyPartExamined': '',
        'segmentationType': 'LABELMAP',
        'segmentAttributes': [segments],
        'ContentLabel': 'SEGMENTATION',
        'ContentDescription': f'{model} multi-label segmentation',
        'ClinicalTrialCoordinatingCenterName': '',
    }

In [ ]:
# ── Stage reference DICOM from GCS (only when input_uri is set) ──
# nb3 needs the source DICOM to build the DICOM-SEG (itkimage2segimage
# --inputDICOMDirectory). For IDC data, download_dicom pulls per-series from IDC.
# For private GCS data the series are not in IDC, so stage them once here with
# s5cmd (mirrors nb1 convertNotebook.ipynb) and sort by SeriesInstanceUID.
if input_uri:
    if not input_uri.startswith('gs://'):
        raise ValueError(f'input_uri must start with gs:// — got {input_uri!r}')
    if not secret_project:
        raise ValueError('secret_project must be set when input_uri is set')
    if GCS_STAGED.exists():
        shutil.rmtree(GCS_STAGED)
    GCS_STAGED.mkdir(parents=True, exist_ok=True)
    bucket, _, prefix = input_uri[len('gs://'):].partition('/')
    print(f'[T+{_elapsed()}] Fetching HMAC credentials (project={secret_project})')
    from google.cloud import secretmanager
    _sm = secretmanager.SecretManagerServiceClient()
    def _secret(sid):
        name = f'projects/{secret_project}/secrets/{sid}/versions/latest'
        return _sm.access_secret_version(name=name).payload.data.decode('utf-8').strip()
    Path('~/.aws').expanduser().mkdir(parents=True, exist_ok=True)
    Path('~/.aws/credentials').expanduser().write_text(
        f"[default]\naws_access_key_id = {_secret('s5cmd-hmac-key-id')}\n"
        f"aws_secret_access_key = {_secret('s5cmd-hmac-secret')}\n")
    _staging = Path('/tmp/gcs_ref_staging')
    if _staging.exists():
        shutil.rmtree(_staging)
    _staging.mkdir(parents=True, exist_ok=True)
    _s3_path = f's3://{bucket}/{prefix}/*' if prefix else f's3://{bucket}/*'
    print(f'[T+{_elapsed()}] s5cmd cp {_s3_path} -> {_staging}', flush=True)
    subprocess.run(['s5cmd', '--endpoint-url', 'https://storage.googleapis.com',
                    'cp', _s3_path, str(_staging) + '/'], check=True)
    import pydicom
    _sort_errors = []
    for _dcm in _staging.rglob('*.dcm'):
        try:
            ds = pydicom.dcmread(str(_dcm), stop_before_pixels=True)
            _d = GCS_STAGED / str(ds.SeriesInstanceUID)
            _d.mkdir(parents=True, exist_ok=True)
            shutil.move(str(_dcm), str(_d / _dcm.name))
        except Exception as exc:
            _sort_errors.append(f'{_dcm}: {exc}')
    shutil.rmtree(_staging, ignore_errors=True)
    _staged = sorted(p.name for p in GCS_STAGED.iterdir() if p.is_dir())
    print(f'[T+{_elapsed()}] Staged reference DICOM for {len(_staged)} series '
          f'({len(_sort_errors)} file(s) failed)')
else:
    print('Reference DICOM source: IDC (per-series download)')

In [ ]:
EXTRACT = Path('/tmp/seg_extract')
DICOM_DIR = Path('/tmp/ref_dicom')
NIFTI_REF_DIR = Path('/tmp/ref_nifti')
DICOM_SEG_DIR = Path('/tmp/dicom_seg')
CONFIG_DIR = Path('/tmp/configs')
RADIOMICS_DIR = Path('/tmp/radiomics')
SR_DICOM_DIR = Path('/tmp/sr_dicom')   # TID1500 SR objects (tid1500writer output)
SR_JSON_DIR = Path('/tmp/sr_json')     # tid1500writer meta-JSON inputs
for d in (EXTRACT, DICOM_DIR, NIFTI_REF_DIR, DICOM_SEG_DIR, CONFIG_DIR, RADIOMICS_DIR,
          SR_DICOM_DIR, SR_JSON_DIR):
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)

subprocess.run(f'lz4 -d -c {segmentationArchivePath} | tar -xf - -C {EXTRACT}',
               shell=True, check=True)
roots = [p for p in EXTRACT.iterdir() if p.is_dir()]
seg_root = roots[0] if len(roots) == 1 and roots[0].name == 'segmentations' else EXTRACT
series_dirs = sorted([p for p in seg_root.iterdir() if p.is_dir()])
print(f'Series to process: {len(series_dirs)}')

# SNOMED mapping source: prefer a table bundled into the archive by nb2 (MOOSE
# bundles moosez's authoritative moose_snomed_mapping.csv at the archive root);
# otherwise use the WDL-fetched snomedMappingPath (the curated TotalSegmentator
# CSV). Loaded here — after extraction — so the bundled copy can take precedence.
_bundled_snomed = seg_root / 'snomed_mapping.csv'
_snomed_source = _bundled_snomed if _bundled_snomed.exists() else Path(snomedMappingPath)
if not _snomed_source.exists():
    raise FileNotFoundError(
        f'No SNOMED mapping found: neither a bundled {_bundled_snomed} nor the '
        f'WDL-fetched snomedMappingPath ({snomedMappingPath}) exists')
label_to_snomed = _load_snomed_csv(str(_snomed_source))
print(f'Loaded {len(label_to_snomed)} SNOMED mappings from {_snomed_source}')

# ---- checkpoint / resume (segmentator_checkpoint.py is fetched next to the notebook by
#      the WDL; no-op when checkpointGcs is empty) ----
import sys
for _p in (os.getcwd(), str(Path.cwd())):
    if _p not in sys.path:
        sys.path.insert(0, _p)
try:
    from segmentator_checkpoint import Checkpointer
except ImportError:
    Checkpointer = None
if checkpointGcs and Checkpointer is None:
    raise RuntimeError('checkpointGcs is set but segmentator_checkpoint.py was not found next to the notebook')
ckpt = Checkpointer(checkpointGcs, runId) if Checkpointer else None
CKPT_ROOTS = {'dicom_seg': DICOM_SEG_DIR, 'radiomics': RADIOMICS_DIR,
              'sr_dicom': SR_DICOM_DIR, 'sr_json': SR_JSON_DIR}
completed_out = ckpt.restore_series_outputs(CKPT_ROOTS) if ckpt else set()
if completed_out:
    print(f'[T+{_elapsed()}] {len(completed_out)} series restored from checkpoint (SEG + radiomics + SR skipped)')

In [ ]:
dicom_seg_errors = []
radiomics_errors = []
sr_errors = []
usage_metrics = {'series': {}}

# -- Radiomics engine config --
# Feature scope comes from the radiomicsFeatureClasses parameter (WDL input),
# normalized in the Parameters cell to canonical names; map to each engine here.
PYRADIOMICS_FEATURE_CLASSES = tuple(RADIOMICS_FEATURE_CLASS_MAP[c][0] for c in radiomicsFeatureClasses)
RADIOMICS_JL_FEATURES = [RADIOMICS_FEATURE_CLASS_MAP[c][1] for c in radiomicsFeatureClasses]
# Radiomics.jl driver script -- fetched next to this notebook by the WDL task; the
# Julia runtime + Radiomics.jl themselves live in the output_conversion image.
# The feature list is sent to the worker with every request ("features": [...]).
RADIOMICS_JL_SCRIPT = os.environ.get('RADIOMICS_JL_SCRIPT', 'radiomics_jl_extract.jl')

extractor = None
if runRadiomics and radiomicsMethod == 'pyradiomics':
    try:
        from radiomics import featureextractor
        extractor = featureextractor.RadiomicsFeatureExtractor()
        extractor.disableAllFeatures()
        for _cls in PYRADIOMICS_FEATURE_CLASSES:
            extractor.enableFeatureClassByName(_cls)
    except Exception as exc:
        radiomics_errors.append(f'pyradiomics init failed: {exc}')
        print(f'WARNING: pyradiomics unavailable: {exc}')
        extractor = None

# radiomics_active: user asked for radiomics AND the selected engine is ready.
radiomics_active = False
if runRadiomics:
    if radiomicsMethod == 'pyradiomics':
        radiomics_active = extractor is not None
    elif radiomicsMethod == 'radiomicsjl':
        radiomics_active = Path(RADIOMICS_JL_SCRIPT).exists()
        if not radiomics_active:
            radiomics_errors.append(f'Radiomics.jl driver not found: {RADIOMICS_JL_SCRIPT}')
            print(f'WARNING: Radiomics.jl driver missing: {RADIOMICS_JL_SCRIPT}')

def _ref_nifti_for(uid, dicom_dest):
    """Fallback: convert the reference DICOM series to a NIfTI intensity volume
    for radiomics. Only used when nb2 did not propagate reference.nii.gz."""
    out = NIFTI_REF_DIR / uid
    out.mkdir(parents=True, exist_ok=True)
    subprocess.run(['dcm2niix', '-z', 'y', '-f', '%s_%d', '-o', str(out), str(dicom_dest)],
                   capture_output=True, text=True)
    cands = sorted(out.glob('*.nii.gz'), key=lambda f: f.stat().st_size, reverse=True)
    return cands[0] if cands else None

# ---- Canonical feature naming ----------------------------------------------
# Both engines emit the SAME row keys: <class>_<snake_case_feature> with the
# canonical class names of RADIOMICS_FEATURE_CLASS_MAP (firstorder, shape, glcm,
# ...), e.g. firstorder_energy, shape_sphericity, glcm_cluster_shade. Engine
# diagnostics (pyradiomics 'diagnostics_*', Radiomics.jl 'diagnosis_*') are not
# features and are dropped.
_CANON_SPECIAL = {'firstorder_10_percentile': 'firstorder_percentile10',
                  'firstorder_90_percentile': 'firstorder_percentile90',
                  # 2D/3D defeat the generic CamelCase->snake_case split
                  'shape_maximum2_d_diameter_column': 'shape_maximum_2d_diameter_column',
                  'shape_maximum2_d_diameter_row': 'shape_maximum_2d_diameter_row',
                  'shape_maximum2_d_diameter_slice': 'shape_maximum_2d_diameter_slice',
                  'shape_maximum3_d_diameter': 'shape_maximum_3d_diameter'}

def _canon_snake(name):
    s = re.sub(r'(?<=[a-z0-9])([A-Z])', r'_\1', name)
    s = re.sub(r'(?<=[A-Z])([A-Z][a-z])', r'_\1', s)
    return s.lower()

def _canon_pyradiomics(key):
    """original_<class>_<CamelCase> -> <class>_<snake_case>; None for non-features."""
    m = re.match(r'^original_([a-z]+)_(.+)$', key)
    if not m:
        return None
    cls, feat = m.group(1), m.group(2)
    if feat[:2] in ('10', '90'):
        feat = feat[:2] + '_' + feat[2:]
    cand = f'{cls}_{_canon_snake(feat)}'
    return _CANON_SPECIAL.get(cand, cand)

def _canon_radiomicsjl(key):
    """Radiomics.jl key -> canonical: first_order_/firstorder_ -> firstorder_,
    shape3d_/shape_ -> shape_ (already snake_case); None for non-features."""
    k = str(key).lower()
    if k.startswith('diagnosis'):
        return None
    for pre, canon in (('first_order_', 'firstorder_'), ('shape3d_', 'shape_')):
        if k.startswith(pre):
            return canon + k[len(pre):]
    return k

# ---- DICOM SR (TID1500) ------------------------------------------------------
# Ported from TotalSegmentator/Notebooks/dicomsegAndRadiomicsSR_Notebook.ipynb.
# radiomicsFeatureCodesPath maps pyradiomics-style feature names to DICOM
# quantity codes (IBSI) + units (UCUM); keys are canonicalized to the same
# <class>_<snake_case> names both engines emit, so the SR writer is engine-
# neutral. Features without a coded row (texture classes, engine extras like
# number_of_islands) get no SR measurement and stay JSON-only.
import pydicom

def _load_feature_codes(path):
    codes = {}
    p = Path(str(path or ''))
    if not p.exists():
        return codes
    with open(p, newline='') as f:
        for row in csv.DictReader(f):
            cls = (row.get('pyradiomics_feature_class') or '').strip()
            feat = (row.get('feature') or '').strip()
            if not cls or not feat:
                continue
            if feat[:2] in ('10', '90'):
                feat = feat[:2] + '_' + feat[2:]
            cand = f'{cls}_{_canon_snake(feat)}'
            quantity = _code(row['quantity_CodingSchemeDesignator'], row['quantity_CodeValue'], row['quantity_CodeMeaning'])
            units = _code(row['units_CodingSchemeDesignator'], row['units_CodeValue'], row['units_CodeMeaning'])
            if quantity and units:
                codes[_CANON_SPECIAL.get(cand, cand)] = {'quantity': quantity, 'units': units}
    return codes

# sr_active: user asked for SRs AND radiomics runs AND the code map + writer exist.
FEATURE_CODES = {}
sr_active = False
if runStructuredReport and radiomics_active:
    FEATURE_CODES = _load_feature_codes(radiomicsFeatureCodesPath)
    if not FEATURE_CODES:
        sr_errors.append(f'feature-codes CSV missing or empty: {radiomicsFeatureCodesPath}; no DICOM SR will be written')
        print(f'WARNING: {sr_errors[-1]}')
    elif shutil.which('tid1500writer') is None:
        sr_errors.append('tid1500writer not found on PATH; no DICOM SR will be written')
        print(f'WARNING: {sr_errors[-1]}')
    else:
        sr_active = True
        print(f'DICOM SR: enabled ({len(FEATURE_CODES)} coded features)')

# Engine identity for the SR observer context / algorithm identification.
if radiomicsMethod == 'pyradiomics':
    try:
        import radiomics as _pyradiomics_pkg
        RADIOMICS_ENGINE = ('pyradiomics', str(_pyradiomics_pkg.__version__))
    except Exception:
        RADIOMICS_ENGINE = ('pyradiomics', 'unknown')
else:
    # Version = the pin baked into the output_conversion image (see
    # workflows/common/Dockerfiles/output_conversion/Dockerfile).
    RADIOMICS_ENGINE = ('Radiomics.jl', os.environ.get('RADIOMICS_JL_VERSION', '1.3.3'))

def _write_structured_report(uid, model, seg_idx, out_dcm, dicom_dest, rows, series_number):
    """One TID1500 SR per SEG object (dcmqi tid1500writer): a measurement group
    per measured label, an item per coded feature. Returns False when nothing was
    codeable (no SR written); raises on tid1500writer failure."""
    seg_sop = str(pydicom.dcmread(str(out_dcm), stop_before_pixels=True).SOPInstanceUID)
    dcm_files = sorted(dicom_dest.rglob('*.dcm'))
    groups = []
    for row in rows:
        if 'radiomics_skipped' in row:
            continue
        rec = label_to_snomed.get(str(row['label_name']).strip())
        if not (rec and rec.get('category') and rec.get('type')):
            continue  # build_dcmqi_config already enforced the mapping; defensive
        items = []
        for key, val in row.items():
            coded = FEATURE_CODES.get(key)
            if coded is None or not isinstance(val, (int, float)) or not np.isfinite(val):
                continue
            # IBSI kurtosis (IPH6) is excess kurtosis; both engines report the
            # Pearson definition, so subtract 3 in the coded SR value only (the
            # radiomics JSON keeps the engine's native value).
            if key == 'firstorder_kurtosis':
                val = val - 3.0
            items.append({'value': str(round(float(val), 3)),
                          'quantity': coded['quantity'], 'units': coded['units']})
        if not items:
            continue
        group = {
            'TrackingIdentifier': f'Measurements group {int(row["label_id"])}',
            # the SEG is written with --useLabelIDAsSegmentNumber, so segment number == label_id
            'ReferencedSegment': int(row['label_id']),
            'SourceSeriesForImageSegmentation': uid,
            'segmentationSOPInstanceUID': seg_sop,
            'Finding': rec['category'],
            'FindingSite': rec['type'],
            'measurementAlgorithmIdentification': {
                'AlgorithmName': RADIOMICS_ENGINE[0],
                'AlgorithmVersion': RADIOMICS_ENGINE[1]},
            'measurementItems': items,
        }
        if rec.get('type_modifier'):
            group['Laterality'] = rec['type_modifier']
        groups.append(group)
    if not groups:
        return False
    meta = {
        '@schema': 'https://raw.githubusercontent.com/qiicr/dcmqi/master/doc/schemas/sr-tid1500-schema.json#',
        'SeriesDescription': f'{model} Radiomics Measurements of series {series_number}',
        'SeriesNumber': str(int(series_number) * 100 + 2) if series_number.isdigit() else '102',
        'InstanceNumber': '1',
        'compositeContext': [out_dcm.name],
        'imageLibrary': [f.name for f in dcm_files],
        'observerContext': {'ObserverType': 'DEVICE',
                            'DeviceObserverName': RADIOMICS_ENGINE[0],
                            'DeviceObserverModelName': RADIOMICS_ENGINE[1]},
        'VerificationFlag': 'UNVERIFIED',
        'CompletionFlag': 'COMPLETE',
        'activitySession': '1',
        'timePoint': '1',
        'Measurements': groups,
    }
    sr_json = SR_JSON_DIR / uid / f'{model}_{seg_idx}_sr.json'
    sr_json.parent.mkdir(parents=True, exist_ok=True)
    sr_json.write_text(json.dumps(meta, indent=2))
    sr_out = SR_DICOM_DIR / uid / f'{model}_{seg_idx}_sr.dcm'
    sr_out.parent.mkdir(parents=True, exist_ok=True)
    # idc-index may nest the series files; tid1500writer resolves imageLibrary
    # names against the given directory, so point it at the dir holding the .dcm.
    image_dir = dcm_files[0].parent if dcm_files else dicom_dest
    res = subprocess.run(['tid1500writer',
                          '--inputMetadata', str(sr_json),
                          '--inputImageLibraryDirectory', str(image_dir),
                          '--inputCompositeContextDirectory', str(out_dcm.parent),
                          '--outputDICOM', str(sr_out)],
                         capture_output=True, text=True)
    if res.returncode != 0 or not sr_out.exists():
        raise RuntimeError(f'tid1500writer rc={res.returncode}\n{(res.stderr or res.stdout)[-2000:]}')
    return True

def _features_pyradiomics(ref_nifti, seg_file, labels, label_names):
    """One row per label: {label_id, label_name, <canonical feature>: float}."""
    rows = []
    for label_id in labels:
        name = label_names.get(str(label_id)) or f'segment_{label_id}'
        feats = extractor.execute(str(ref_nifti), str(seg_file), label=int(label_id))
        row = {'label_id': label_id, 'label_name': name}
        for k, v in feats.items():
            canon = _canon_pyradiomics(k)
            if canon is None:
                continue
            # pyradiomics returns most feature values as 0-d numpy arrays, for
            # which np.isscalar() is False -- test ndim == 0, not scalar-ness.
            arr = np.asarray(v)
            if arr.ndim == 0:
                try:
                    row[canon] = float(arr)
                except (TypeError, ValueError):
                    pass
        rows.append(row)
    return rows

# -- Radiomics.jl persistent worker -----------------------------------------
# Julia startup + package load + JIT costs ~8-10 s per process. The pilot runs
# showed MOOSE paying that 10x per series (one seg file per sub-model), so nb3
# keeps ONE `julia --worker` process for the whole run and sends it a JSON
# request per seg file (all labels at once). Protocol: see radiomics_jl_extract.jl.
# Started with -t <radiomicsJlThreads> (0 = auto = all vCPUs) so Radiomics.jl can
# parallelise across labels on the 4-vCPU output-conversion VM.
_JL_SENTINEL = '@@RESULT '
_jl_worker = {'proc': None, 'next_id': 0, 'restarts': 0}

def _jl_start():
    threads = str(int(radiomicsJlThreads)) if int(radiomicsJlThreads or 0) > 0 else 'auto'
    proc = subprocess.Popen(['julia', '-t', threads, RADIOMICS_JL_SCRIPT, '--worker'],
                            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            text=True, bufsize=1)
    _jl_worker['proc'] = proc
    print(f'  [radiomicsjl] worker started (pid {proc.pid}, threads={threads}, '
          f'features={RADIOMICS_JL_FEATURES})')
    return proc

def _jl_stop():
    proc = _jl_worker.get('proc')
    if proc is not None and proc.poll() is None:
        try:
            proc.stdin.close()
            proc.wait(timeout=60)
        except Exception:
            proc.kill()
    _jl_worker['proc'] = None

_JL_MAX_RESTARTS = 10   # a crashing series costs 2 restarts (request is retried once)

def _jl_reap(proc, why):
    """The worker died (or its pipe broke): collect it, log stderr, forget it."""
    try:
        proc.wait(timeout=10)
    except Exception:
        proc.kill()
    err = ''
    try:
        err = proc.stderr.read()[-2000:] if proc.stderr else ''
    except Exception:
        pass
    _jl_worker['restarts'] += 1
    print(f'  [radiomicsjl] worker died ({why}, rc={proc.returncode}); '
          f'restart {_jl_worker["restarts"]}/{_JL_MAX_RESTARTS}. stderr tail:\n{err}')
    _jl_worker['proc'] = None

def _jl_request(ref_nifti, seg_file, labels):
    """One request/response round trip; if the worker dies, it is restarted and the
    request retried once (a request that kills the worker twice raises)."""
    for attempt in (1, 2):
        proc = _jl_worker.get('proc')
        if proc is not None and proc.poll() is not None:
            _jl_reap(proc, 'exited between requests')
            proc = None
        if proc is None:
            if _jl_worker['restarts'] >= _JL_MAX_RESTARTS:
                raise RuntimeError('Radiomics.jl worker died repeatedly; giving up on radiomicsjl')
            proc = _jl_start()
        _jl_worker['next_id'] += 1
        rid = _jl_worker['next_id']
        req = {'id': rid, 'ref': str(ref_nifti), 'seg': str(seg_file),
               'labels': [int(l) for l in labels],
               'features': RADIOMICS_JL_FEATURES}
        try:
            proc.stdin.write(json.dumps(req) + '\n')
            proc.stdin.flush()
            while True:
                line = proc.stdout.readline()
                if not line:            # EOF: worker died mid-request
                    raise BrokenPipeError('worker closed stdout')
                if line.startswith(_JL_SENTINEL):
                    resp = json.loads(line[len(_JL_SENTINEL):])
                    if resp.get('id') != rid:
                        continue        # stale response from a previous (failed) request
                    if 'error' in resp:
                        raise RuntimeError(f'radiomics_jl_extract.jl: {resp["error"]}')
                    return resp.get('result', [])
                # any other stdout chatter from the library: ignore
        except (BrokenPipeError, OSError, ValueError) as exc:
            # ValueError: 'I/O operation on closed file' when the pipe went away
            _jl_reap(proc, f'{type(exc).__name__} during request {rid}')
            if attempt == 2:
                raise RuntimeError(f'Radiomics.jl worker crashed twice on {seg_file}: {exc}')

def _features_radiomicsjl(ref_nifti, seg_file, labels, label_names):
    """All labels of one seg file in ONE request to the persistent Julia worker.
    The worker returns a JSON array of {label_id, <feature>: value}; label_name
    is attached here."""
    by_label = {int(r['label_id']): r for r in _jl_request(ref_nifti, seg_file, labels)}
    rows = []
    for label_id in labels:
        name = label_names.get(str(label_id)) or f'segment_{label_id}'
        feats = by_label.get(int(label_id), {})
        row = {'label_id': label_id, 'label_name': name}
        row.update({_canon_radiomicsjl(k): float(v) for k, v in feats.items()
                    if k != 'label_id' and isinstance(v, (int, float))
                    and _canon_radiomicsjl(k) is not None})
        rows.append(row)
    return rows

_RADIOMICS_BACKENDS = {
    'pyradiomics': _features_pyradiomics,
    'radiomicsjl': _features_radiomicsjl,
}

def extract_features(ref_nifti, seg_file, labels, label_names, uid, model):
    """Engine-agnostic entry point. Dispatches on radiomicsMethod and stamps the
    shared identifying fields so downstream JSON/packaging is engine-independent."""
    rows = []
    for fr in _RADIOMICS_BACKENDS[radiomicsMethod](ref_nifti, seg_file, labels, label_names):
        row = {'SeriesInstanceUID': uid, 'model': model,
               'radiomics_method': radiomicsMethod,
               'label_id': fr['label_id'], 'label_name': fr['label_name']}
        row.update({k: v for k, v in fr.items() if k not in ('label_id', 'label_name')})
        rows.append(row)
    return rows

for series_dir in series_dirs:
    uid = series_dir.name
    print(f'\n=== {uid} ===')
    if uid in completed_out:
        usage_metrics['series'].setdefault(uid, {})['checkpoint_restored'] = True
        print('  restored from checkpoint')
        continue
    t_series = time.time()
    dicom_dest = DICOM_DIR / uid
    t_dl = time.time()
    try:
        download_dicom(uid, dicom_dest)
        usage_metrics['series'].setdefault(uid, {})['ref_download_s'] = round(time.time()-t_dl, 1)
    except Exception as exc:
        dicom_seg_errors.append(f'{uid}: reference DICOM download failed: {exc}')
        print(f'  ERROR downloading reference DICOM: {exc}')
        continue
    series_number = find_series_number(dicom_dest)
    # Radiomics reference: prefer the exact NIfTI nb2 propagated to the series
    # root (identical geometry to the mask, so the engine never fails on a
    # geometry mismatch); fall back to converting the reference DICOM for
    # archives produced before reference.nii.gz existed.
    ref_nifti = None
    if radiomics_active:
        _propagated = series_dir / 'reference.nii.gz'
        ref_nifti = _propagated if _propagated.exists() else _ref_nifti_for(uid, dicom_dest)

    for model_dir in sorted([p for p in series_dir.iterdir() if p.is_dir()]):
        model = model_dir.name
        lm_path = model_dir / 'label_map.json'
        label_names = {}
        if lm_path.exists():
            label_names = json.loads(lm_path.read_text()).get('labels', {})
        seg_files = sorted((model_dir / 'segmentations').glob('*.nii.gz')) \
            if (model_dir / 'segmentations').exists() else sorted(model_dir.rglob('*.nii.gz'))
        for seg_idx, seg_file in enumerate(seg_files):
            try:
                label_counts = label_voxel_counts(seg_file)
                labels = sorted(label_counts)
                if not labels:
                    print(f'  {model}/{seg_file.name}: empty mask, skipping')
                    continue
                # -- DICOM-SEG --
                cfg = build_dcmqi_config(model, labels, label_names, series_number)
                cfg_path = CONFIG_DIR / f'{uid}_{model}_{seg_idx}.json'
                cfg_path.write_text(json.dumps(cfg, indent=2))
                out_dir = DICOM_SEG_DIR / uid
                out_dir.mkdir(parents=True, exist_ok=True)
                out_dcm = out_dir / f'{model}_{seg_idx}.dcm'
                t0 = time.time()
                res = subprocess.run([
                    'itkimage2segimage',
                    '--inputImageList', str(seg_file),
                    '--inputDICOMDirectory', str(dicom_dest),
                    '--outputDICOM', str(out_dcm),
                    '--inputMetadata', str(cfg_path),
                    '--segmentationType', 'labelmap',
                    '--useLabelIDAsSegmentNumber',
                    '--skip', '1',
                    '--compress', 'deflate',
                ], capture_output=True, text=True)
                if res.returncode != 0 or not out_dcm.exists():
                    dicom_seg_errors.append(
                        f'{uid}/{model}/{seg_file.name}: itkimage2segimage rc={res.returncode}\n{res.stderr}')
                    print(f'  ERROR SEG {model}: rc={res.returncode}')
                else:
                    usage_metrics['series'].setdefault(uid, {}).setdefault('seg_s', {})[model] = round(time.time()-t0, 1)
                    print(f'  {model}: wrote {out_dcm.name} ({len(labels)} segments)')
                # -- radiomics (engine selected by radiomicsMethod) --
                usage_metrics['series'].setdefault(uid, {}).setdefault('n_labels', {})[model] = len(labels)
                if radiomics_active and ref_nifti is not None:
                    t_rad = time.time()
                    # ROI-size guard: radiomics cost scales with ROI voxels (first-order is
                    # O(N), 3D shape meshes the surface). A whole-body mask (MOOSE
                    # clin_ct_body, ~10-60 Mvox) took ~1170 s/series in the pilot vs <70 s
                    # for any organ model, so labels above radiomicsMaxRoiMvox are skipped
                    # and recorded (SEG is still written for them).
                    _max_vox = float(radiomicsMaxRoiMvox or 0) * 1e6
                    big = [l for l in labels if _max_vox > 0 and label_counts[l] > _max_vox]
                    rad_labels = [l for l in labels if l not in big]
                    if big:
                        usage_metrics['series'].setdefault(uid, {}).setdefault('n_labels_skipped_large', {})[model] = len(big)
                        print(f'  {model}: radiomics skipped for {len(big)} label(s) with ROI > '
                              f'{radiomicsMaxRoiMvox} Mvox: '
                              + ', '.join(f"{l}({label_names.get(str(l), l)}:{label_counts[l]/1e6:.1f}M)" for l in big))
                    rows = []
                    try:
                        rows = extract_features(ref_nifti, seg_file, rad_labels, label_names, uid, model) if rad_labels else []
                        for l in big:
                            rows.append({'SeriesInstanceUID': uid, 'model': model,
                                         'radiomics_method': radiomicsMethod,
                                         'label_id': l, 'label_name': label_names.get(str(l)) or f'segment_{l}',
                                         'radiomics_skipped': f'roi_voxels {label_counts[l]} > '
                                                              f'radiomicsMaxRoiMvox {radiomicsMaxRoiMvox} Mvox'})
                        rad_path = RADIOMICS_DIR / uid / f'{model}_{seg_idx}.json'
                        rad_path.parent.mkdir(parents=True, exist_ok=True)
                        rad_path.write_text(json.dumps(rows, indent=2))
                    except Exception as exc:
                        radiomics_errors.append(f'{uid}/{model}: {exc}')
                        print(f'  WARNING radiomics {model}: {exc}')
                    finally:
                        _rs = usage_metrics['series'].setdefault(uid, {}).setdefault('radiomics_s', {})
                        _rs[model] = round(_rs.get(model, 0) + time.time() - t_rad, 1)
                    # -- DICOM SR (TID1500): needs both the SEG object and the rows --
                    if sr_active and out_dcm.exists() and rows:
                        t_sr = time.time()
                        try:
                            if _write_structured_report(uid, model, seg_idx, out_dcm, dicom_dest, rows, series_number):
                                print(f'  {model}: wrote SR {model}_{seg_idx}_sr.dcm')
                        except Exception as exc:
                            sr_errors.append(f'{uid}/{model}: {exc}')
                            print(f'  WARNING SR {model}: {exc}')
                        finally:
                            _ss = usage_metrics['series'].setdefault(uid, {}).setdefault('sr_s', {})
                            _ss[model] = round(_ss.get(model, 0) + time.time() - t_sr, 1)
            except Exception as exc:
                dicom_seg_errors.append(f'{uid}/{model}/{seg_file.name}: {traceback.format_exc()}')
                print(f'  ERROR {model}: {exc}')
    usage_metrics['series'].setdefault(uid, {})['total_s'] = round(time.time()-t_series, 1)
    shutil.rmtree(dicom_dest, ignore_errors=True)
    if ckpt:
        ckpt.save_series_output(uid, CKPT_ROOTS)

if radiomicsMethod == 'radiomicsjl':
    _jl_stop()

if dicom_seg_errors:
    Path('dicom_seg_error_file.txt').write_text('\n\n'.join(dicom_seg_errors))
if radiomics_errors:
    Path('radiomics_error_file.txt').write_text('\n\n'.join(radiomics_errors))
if sr_errors:
    Path('sr_error_file.txt').write_text('\n\n'.join(sr_errors))
print(f'\n[T+{_elapsed()}] SEG/radiomics/SR complete ({len(dicom_seg_errors)} SEG, '
      f'{len(radiomics_errors)} radiomics, {len(sr_errors)} SR error(s))')

## (Optional) Upload DICOM-SEG to GCS and import into a Healthcare DICOM store

In [ ]:
if dicomSegBucketUri:
    try:
        from google.cloud import storage
        bkt_name, _, prefix = dicomSegBucketUri[len('gs://'):].partition('/')
        bucket = storage.Client().bucket(bkt_name)
        for dcm in DICOM_SEG_DIR.rglob('*.dcm'):
            rel = dcm.relative_to(DICOM_SEG_DIR)
            blob_name = f"{prefix.rstrip('/')}/{rel}".lstrip('/')
            bucket.blob(blob_name).upload_from_filename(str(dcm))
        print(f'Uploaded DICOM-SEG to {dicomSegBucketUri}')
    except Exception as exc:
        print(f'WARNING: GCS upload failed: {exc}')

if dicomStoreImportUri and dicomSegBucketUri:
    try:
        import google.auth
        from google.auth.transport.requests import AuthorizedSession
        creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
        session = AuthorizedSession(creds)
        gcs_source = f"{dicomSegBucketUri.rstrip('/')}/**.dcm"
        url = f'https://healthcare.googleapis.com/v1/{dicomStoreImportUri}:import'
        body = {'gcsSource': {'uri': gcs_source}}
        resp = session.post(url, json=body)
        resp.raise_for_status()
        op = resp.json().get('name')
        print(f'Started dicomStores.import operation: {op}')
        deadline = time.time() + 1800
        while time.time() < deadline:
            st = session.get(f'https://healthcare.googleapis.com/v1/{op}').json()
            if st.get('done'):
                print('Import done:', json.dumps(st.get('error', st.get('metadata', {})), indent=2))
                break
            time.sleep(15)
    except Exception as exc:
        print(f'WARNING: Healthcare import failed: {exc}')

## Package outputs + combined usage metrics + run summary

In [ ]:
# DICOM-SEG archive (required output)
if not any(DICOM_SEG_DIR.rglob('*.dcm')):
    raise RuntimeError('No DICOM-SEG files produced — see dicom_seg_error_file.txt')
subprocess.run(f'tar -cf - -C {DICOM_SEG_DIR.parent} {DICOM_SEG_DIR.name} | lz4 > dicom_seg.tar.lz4',
               shell=True, check=True)

# Radiomics archive (optional)
if runRadiomics and any(RADIOMICS_DIR.rglob('*.json')):
    subprocess.run(f'tar -cf - -C {RADIOMICS_DIR.parent} {RADIOMICS_DIR.name} | lz4 > radiomics.tar.lz4',
                   shell=True, check=True)

# DICOM SR archives (optional): the TID1500 SR objects plus the tid1500writer
# meta-JSONs they were built from (the measurements in schema-shaped JSON).
if runStructuredReport and any(SR_DICOM_DIR.rglob('*.dcm')):
    subprocess.run(f'tar -cf - -C {SR_DICOM_DIR.parent} {SR_DICOM_DIR.name} | lz4 > structured_reports_dicom.tar.lz4',
                   shell=True, check=True)
if runStructuredReport and any(SR_JSON_DIR.rglob('*.json')):
    subprocess.run(f'tar -cf - -C {SR_JSON_DIR.parent} {SR_JSON_DIR.name} | lz4 > structured_reports_json.tar.lz4',
                   shell=True, check=True)

# Usage metrics for this task
# Per-series phase timings: SEG (dcmqi), radiomics and SR per model, reference-DICOM
# download and total per series, plus label count -- so download / SEG / radiomics
# can be separated when profiling (see util/executionAnalytics/README.md).
with open('output_conversion_UsageMetrics.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['SeriesInstanceUID', 'model', 'model_seg_s', 'model_radiomics_s', 'model_sr_s',
                'n_labels', 'n_labels_skipped_large', 'ref_download_s', 'series_total_s',
                'radiomics_method', 'checkpoint_restored'])
    for uid, m in usage_metrics['series'].items():
        seg_s = m.get('seg_s', {})
        rad_s = m.get('radiomics_s', {})
        sr_s = m.get('sr_s', {})
        n_lab = m.get('n_labels', {})
        n_skip = m.get('n_labels_skipped_large', {})
        for model in (list(seg_s) or list(rad_s) or ['']):
            w.writerow([uid, model, seg_s.get(model, ''), rad_s.get(model, ''), sr_s.get(model, ''),
                        n_lab.get(model, ''),
                        n_skip.get(model, 0) if radiomics_active else '',
                        m.get('ref_download_s', ''), m.get('total_s', ''),
                        radiomicsMethod if radiomics_active else '',
                        bool(m.get('checkpoint_restored', False))])

# Combined metrics = inference CSV (if provided) + this task's CSV, concatenated.
with open('combined_UsageMetrics.csv', 'w', newline='') as out:
    for src in [convertUsageMetricsCsvPath, inferenceUsageMetricsCsvPath,
                'output_conversion_UsageMetrics.csv']:
        if src and Path(src).exists():
            out.write(f'# source: {Path(src).name}\n')
            out.write(Path(src).read_text())
            out.write('\n')

run_summary = {
    'runId': runId,
    'model': modelName,
    'series_count': len(series_dirs),
    'dicom_seg_errors': len(dicom_seg_errors),
    'radiomics_errors': len(radiomics_errors),
    'sr_errors': len(sr_errors),
    'radiomics_enabled': bool(runRadiomics),
    'radiomics_method': radiomicsMethod if runRadiomics else None,
    'radiomics_feature_classes': list(radiomicsFeatureClasses) if runRadiomics else None,
    'radiomics_jl_threads': (int(radiomicsJlThreads) if int(radiomicsJlThreads or 0) > 0 else 'auto')
        if runRadiomics and radiomicsMethod == 'radiomicsjl' else None,
    'radiomics_max_roi_mvox': float(radiomicsMaxRoiMvox or 0) if runRadiomics else None,
    'radiomics_labels_skipped_large': int(sum(sum(v.values()) for v in
        (m.get('n_labels_skipped_large', {}) for m in usage_metrics['series'].values()))),
    'structured_report_enabled': bool(runStructuredReport),
    'structured_reports_written': sum(1 for _ in SR_DICOM_DIR.rglob('*.dcm')),
    'total_elapsed_s': round(time.time() - NOTEBOOK_START, 1),
}
Path('run_summary.json').write_text(json.dumps(run_summary, indent=2))
print(json.dumps(run_summary, indent=2))

# Outputs are packaged: the checkpoint is no longer needed.
if ckpt:
    ckpt.cleanup()
print(f'[T+{_elapsed()}] Output conversion complete')